# Phase 6 — Prompt Caching

Every call you've made so far re-processes the entire input from scratch. For a chatbot that has a 2-paragraph system prompt + 3 tool schemas, that's the same 1500+ tokens being re-tokenized and re-attended on every turn. Wasteful.

Prompt caching tells the API: "this prefix is identical to last time — reuse the cached processing." Result: ~90% input-token cost reduction on the cached portion, and noticeably faster responses.

## What you'll learn
1. **Why caching exists** — input processing is expensive and often identical between calls
2. **Cache breakpoints** — `cache_control: {type: "ephemeral"}` markers
3. **The order** — tools → system → messages, and what "cache up to here" means
4. **Reading the usage object** — `cache_creation_input_tokens` vs `cache_read_input_tokens`
5. **Cache invalidation** — what breaks the cache (and what doesn't)

## Setup

In [ ]:
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

import json
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-5"

## Step 1 — A request big enough to cache

**Important rule:** content under 1024 tokens won't cache. So we need a chunky system prompt to see anything happen. Below is an intentionally long support-bot system prompt — pretend it's your real product's instructions.

In [ ]:
# A long-ish system prompt — repeated for length to easily clear the 1024-token threshold.
BIG_SYSTEM_PROMPT = (
    "You are Atlas, the official support assistant for SkyForge Cloud, a platform-as-a-service "
    "company. Your job is to help users with billing, account, deployment, and technical questions.\n\n"
    "Tone: warm but professional. Always greet the user by name if known. Use short paragraphs. "
    "Never make up information you don't have — if you're unsure, say so and offer to escalate.\n\n"
    "Product overview:\n"
    "- SkyForge Compute = managed VM instances, billed per second of runtime\n"
    "- SkyForge Storage = S3-compatible object storage, billed per GB-month\n"
    "- SkyForge Edge = CDN with 240 PoPs worldwide, billed per GB transferred\n"
    "- SkyForge Vector = managed vector DB, billed per million vectors stored\n\n"
    "Pricing tiers: Free (1 project, 100GB), Pro ($29/mo, 10 projects, 1TB), "
    "Business ($299/mo, unlimited projects, 10TB, SSO), Enterprise (custom).\n\n"
    "Support escalation paths:\n"
    "- Billing disputes → finance@skyforge.example, response within 1 business day\n"
    "- Outage/SLA claims → status.skyforge.example then sla@skyforge.example\n"
    "- Security/abuse reports → security@skyforge.example, 24/7\n"
    "- Sales/enterprise upgrades → sales@skyforge.example\n\n"
    "Refund policy: Pro and Business plans refundable within 7 days of charge if no resources "
    "were provisioned during the period. Enterprise contracts are non-refundable. Free tier has "
    "no refunds because no payment is collected.\n\n"
    "Deployment regions: us-east-1, us-west-2, eu-west-1, eu-central-1, ap-southeast-1, "
    "ap-northeast-1. New regions added quarterly. Cross-region replication is Pro tier and above.\n\n"
    "Common pitfalls users hit:\n"
    "1. They forget Compute instances bill while stopped if storage is attached — they must detach "
    "and delete the volume to fully stop billing.\n"
    "2. They confuse Edge GB-transferred (egress only) with Storage GB-month (data at rest).\n"
    "3. They try to use Free tier for production and hit the 100GB cap.\n"
    "4. They forget SSO is Business-tier and above.\n\n"
    "Always end with: 'Anything else I can help with?'"
) * 2   # double it just to comfortably exceed 1024 tokens

print(f"System prompt size: ~{len(BIG_SYSTEM_PROMPT)//4} tokens (very rough)")

## Step 2 — Send WITHOUT caching, see the cost

Two back-to-back questions. Look at `usage.input_tokens` — same big number both times because nothing is reused.

In [ ]:
def chat_no_cache(user_text: str):
    response = client.messages.create(
        model=model,
        max_tokens=300,
        system=BIG_SYSTEM_PROMPT,  # plain string — no cache_control
        messages=[{"role": "user", "content": user_text}],
    )
    print("Reply:", response.content[0].text[:120], "...")
    print("Usage:", response.usage)
    return response

print("=== Call 1 ===")
chat_no_cache("Hi! How do I stop being billed for a Compute instance I'm not using?")
print("\n=== Call 2 ===")
chat_no_cache("What's the refund window for Pro tier?")

Both calls show high `input_tokens` and `cache_creation_input_tokens=0`, `cache_read_input_tokens=0`. The big system prompt was re-processed twice.

## Step 3 — Enable caching on the system prompt

Two changes from above:
1. Pass `system` as a **list of blocks** instead of a plain string
2. Add `cache_control: {"type": "ephemeral"}` to the block you want cached

In [ ]:
def chat_with_cache(user_text: str):
    response = client.messages.create(
        model=model,
        max_tokens=300,
        system=[{
            "type": "text",
            "text": BIG_SYSTEM_PROMPT,
            "cache_control": {"type": "ephemeral"}
        }],
        messages=[{"role": "user", "content": user_text}],
    )
    u = response.usage
    print(f"input_tokens={u.input_tokens}  "
          f"cache_creation={u.cache_creation_input_tokens}  "
          f"cache_read={u.cache_read_input_tokens}")
    return response

print("=== Call 1 (writes the cache) ===")
chat_with_cache("How do I stop billing on an idle Compute instance?")

print("\n=== Call 2 (should hit cache) ===")
chat_with_cache("What's the refund window for Pro?")

print("\n=== Call 3 (should also hit cache) ===")
chat_with_cache("Is SSO available on Free tier?")

You should see:
- **Call 1**: `cache_creation_input_tokens` is large (the system prompt got written to cache), `cache_read=0`, `input_tokens` is small (just the new user message).
- **Calls 2 & 3**: `cache_creation=0`, `cache_read_input_tokens` is large (system prompt reused from cache), `input_tokens` is small.

Cache reads are billed at ~10% of normal input cost. Cache writes are ~125% (a one-time tax). So caching pays off after just 2 reuses.

## Step 4 — Cache tool schemas too (for the reminder bot)

Tool schemas live before the system prompt in the processing order:

    tools → system → messages

To cache the tool schemas, attach `cache_control` to the **last tool schema** in the list. Everything up to and including that breakpoint gets cached together.

Below: a mini version of the reminder bot from Phase 5, with caching enabled on both the tools and the system prompt.

In [ ]:
TOOLS = [
    {
        "name": "get_current_datetime",
        "description": "Returns the current date/time. Use whenever the user mentions 'now', 'today', 'this week'.",
        "input_schema": {
            "type": "object",
            "properties": {"date_format": {"type": "string"}},
            "required": ["date_format"]
        }
    },
    {
        "name": "add_duration_to_datetime",
        "description": "Adds days/hours to a base datetime. Use for any date math.",
        "input_schema": {
            "type": "object",
            "properties": {
                "base_datetime": {"type": "string"},
                "days": {"type": "integer"},
                "hours": {"type": "integer"}
            },
            "required": ["base_datetime"]
        }
    },
    {
        "name": "set_reminder",
        "description": "Stores a reminder with a message and target datetime.",
        "input_schema": {
            "type": "object",
            "properties": {
                "message": {"type": "string"},
                "when": {"type": "string"}
            },
            "required": ["message", "when"]
        }
    }
]

# Best practice: COPY the tools list, then add cache_control to the last one.
# Don't mutate the original — you may want to reuse it without caching elsewhere.
tools_with_cache = [dict(t) for t in TOOLS]
tools_with_cache[-1] = {**tools_with_cache[-1], "cache_control": {"type": "ephemeral"}}

print("Last tool schema now has cache_control:")
print(json.dumps(tools_with_cache[-1], indent=2))

**Note:** tool schemas here probably *don't* meet the 1024-token minimum on their own. In production with 10+ rich tool schemas, they easily do. For this demo, the system prompt is the heavy lifter — we still set the tool breakpoint to show the syntax.

## Step 5 — Run a multi-turn conversation with caching

Three turns. Watch how the cache shifts from "creation" to "read" as the conversation stays consistent at the front.

In [ ]:
def cached_call(messages):
    response = client.messages.create(
        model=model,
        max_tokens=300,
        tools=tools_with_cache,
        system=[{
            "type": "text",
            "text": BIG_SYSTEM_PROMPT,
            "cache_control": {"type": "ephemeral"}
        }],
        messages=messages,
    )
    u = response.usage
    print(f"  input={u.input_tokens}  cache_creation={u.cache_creation_input_tokens}  cache_read={u.cache_read_input_tokens}")
    return response

messages = []

for question in [
    "How much does Storage cost?",
    "And Edge?",
    "Can I get a refund if I cancel Pro after 5 days?",
]:
    print(f"\n--- Turn: {question!r} ---")
    messages.append({"role": "user", "content": question})
    response = cached_call(messages)
    # Save reply text and continue (simplified — ignoring tool_use for this demo)
    reply_text = next((b.text for b in response.content if b.type == "text"), "")
    messages.append({"role": "assistant", "content": reply_text})

Reading the numbers:
- Turn 1: big `cache_creation`, zero `cache_read`. The cache was just written.
- Turn 2: zero `cache_creation`, big `cache_read`. The system prompt + tools came from cache; only the small new user/assistant exchange was new input.
- Turn 3: same as turn 2 — bigger `cache_read` now because turn 2's messages also got included in the prefix.

## Step 6 — Cache invalidation

Cache hits require an **exact byte-for-byte match** of everything before (and including) the breakpoint. Change anything in the system prompt → cache miss → you pay the creation cost again.

Watch what happens when we modify the system prompt slightly.

In [ ]:
modified_prompt = BIG_SYSTEM_PROMPT + "\n\nNote: ALL CAPS the company name when mentioning it."

response = client.messages.create(
    model=model,
    max_tokens=300,
    system=[{
        "type": "text",
        "text": modified_prompt,
        "cache_control": {"type": "ephemeral"}
    }],
    messages=[{"role": "user", "content": "What does SkyForge offer?"}],
)
u = response.usage
print(f"After adding one line to system prompt:")
print(f"  cache_creation={u.cache_creation_input_tokens}  (had to rebuild the cache)")
print(f"  cache_read={u.cache_read_input_tokens}  (zero — old cache invalid)")

Cache rebuilt from scratch. Lesson: keep your cached prefixes **stable**. Don't interpolate a timestamp into your system prompt. Don't shuffle tool schema order between calls.

## Rules of caching — the cheat sheet

| Rule | Detail |
|------|--------|
| **Minimum size** | 1024 tokens before the breakpoint (else nothing cached, no error) |
| **Cache lifetime** | ~5 minutes by default (ephemeral); refreshed on every hit. Max 1 hour. |
| **Max breakpoints** | 4 per request. Multiple = multiple cache layers. |
| **Processing order** | tools → system → messages. Cache scope = everything UP TO AND INCLUDING the breakpoint. |
| **Invalidation** | Any byte change before the breakpoint invalidates the cache. |
| **Multi-layer trick** | Put one breakpoint on tools, one on system. If only messages change, both hit. If tools change, system layer still hits independently. |

## Mini-exercise

1. **Two breakpoints.** Add `cache_control` to BOTH the last tool schema AND the system prompt. Run a turn. Then *modify* one of the tool descriptions and run again. You should see the tool layer invalidate but — wait, can the system prompt layer still hit? Try it and check the numbers.
2. **Cache the last message too.** For super-long conversations, put a breakpoint on a `tool_result` block deep in the history. Watch how more of the history starts coming from cache as the conversation grows.
3. **Measure the savings.** From step 3 of this notebook, compare `input_tokens + cache_creation*1.25 + cache_read*0.1` across the 3 turns vs. what you'd pay without caching (`input_tokens` only, but big). For a 10-turn conversation that's the actual ROI calculation.

Tell me the `cache_creation` and `cache_read` numbers you saw on step 5 turn 2, and we'll go to **Phase 7: RAG** — chunk, embed, search, retrieve.